# Chronos-2 Inference Speed Profiling

- **Baseline** — `Chronos2Pipeline.predict()`: the standard high-level API with full dataset/DataLoader machinery

## 1. Config

In [7]:
import math
from pathlib import Path
import time
import warnings
warnings.filterwarnings('ignore')

import os
os.environ.setdefault("HOME", os.environ.get("USERPROFILE", str(Path.home())))

import torch
import numpy as np
from torch.utils.data import DataLoader

from chop.models.chronos2.pipeline import Chronos2Pipeline
from chronos.chronos2.dataset import Chronos2Dataset, DatasetMode

# ── Config ────────────────────────────────────────────────────────────────────
DEVICE           = 'cuda' if torch.cuda.is_available() else 'cpu'
SEQ_LEN          = 1440   # context length per time series
N_FEATURES       = 20     # number of variates per sample
FORECAST_HORIZON = 96     # prediction length
BATCH_SIZES      = [20, 40, 60, 80, 100]  # number of multivariate samples per batch
WARMUP_RUNS      = 3
BENCH_RUNS       = 5
MODEL_ID         = 'amazon/chronos-2'

print(f'Device: {DEVICE}')
print(f'Sequence length: {SEQ_LEN}')
print(f'Number of features: {N_FEATURES}')
print(f'Forecast horizon: {FORECAST_HORIZON}')

Device: cuda
Sequence length: 1440
Number of features: 20
Forecast horizon: 96


## 2. Load model

In [8]:
from chop.models import get_model

model = get_model("chronos-2", pretrained=True, model_id=MODEL_ID)
model = model.to(DEVICE).to(torch.bfloat16)
pipeline = Chronos2Pipeline(model=model)
model.eval()

OUTPUT_PATCH_SIZE = model.chronos_config.output_patch_size
NUM_OUTPUT_PATCHES = math.ceil(FORECAST_HORIZON / OUTPUT_PATCH_SIZE)
FUTURE_LEN = NUM_OUTPUT_PATCHES * OUTPUT_PATCH_SIZE

print(f'✓ Model loaded successfully from {MODEL_ID}')
print(f'  Model type:        {type(model).__name__}')
print(f'  Output patch size: {OUTPUT_PATCH_SIZE}')
print(f'  Num output patches for horizon={FORECAST_HORIZON}: {NUM_OUTPUT_PATCHES}')


✓ Model loaded successfully from amazon/chronos-2
  Model type:        Chronos2Model
  Output patch size: 16
  Num output patches for horizon=96: 6


# 3. Benchmark with fev bench


In [9]:
# Benchmark function 

import pandas as pd


def fev_bench_inference_time(model, model_name, task_configs, batch_sizes, output_dir="artifacts"):
    import fev

    # Define benchmark
    benchmark = fev.Benchmark.from_list(task_configs)

    # Run benchmark for each model and batch size
    summaries = []
    inference_times = {}
    print(f'\n=== Benchmarking model: {model_name} ===')

    # Create pipeline for the model
    pipeline = Chronos2Pipeline(model=model)

    for task in benchmark.tasks:
        print(f'\n--- Task: {task.task_name} ---')

        inference_times[task.task_name] = {}

        for batch_size in batch_sizes:
            predictions, inference_time = pipeline.predict_fev(task, batch_size)
            summary = task.evaluation_summary(predictions, model_name)
            summaries.append(summary)

            inference_times[task.task_name][batch_size] = inference_time
            
            inference_time_per_sample = inference_time / batch_size
            print(f'Batch size: {batch_size} | Inference time per sample: {inference_time_per_sample:.4f} sec')

    # ── Save results to CSV ───────────────────────────────────────────────────
    Path(output_dir).mkdir(parents=True, exist_ok=True)

    # Summaries CSV
    summaries_df = pd.DataFrame(summaries)
    summaries_path = Path(output_dir) / f"{model_name}_summaries.csv"
    summaries_df.to_csv(summaries_path, index=False)
    print(f'\nSaved summaries → {summaries_path}')

    # Inference times CSV  (rows = tasks, columns = batch sizes)
    inference_rows = []
    for task_name, batch_dict in inference_times.items():
        for batch_size, t in batch_dict.items():
            inference_rows.append({
                "task": task_name,
                "batch_size": batch_size,
                "inference_time_s": t,
                "inference_time_per_sample_s": t / batch_size,
            })
    inference_df = pd.DataFrame(inference_rows)
    inference_path = Path(output_dir) / f"{model_name}_inference_times.csv"
    inference_df.to_csv(inference_path, index=False)
    print(f'Saved inference times → {inference_path}')

    return summaries_df, inference_df


In [17]:
# Benchmark on baseline chronos2 
tasks_configs = [
    {
        "dataset_path": "autogluon/chronos_datasets",
        "dataset_config": "m4_hourly",
        "horizon": 24,
    },
]

BATCH_SIZES = [20, 40, 60, 80, 100]

fev_bench_inference_time(
    model=model,
    model_name="Baseline",
    task_configs=tasks_configs,
    batch_sizes=BATCH_SIZES,
)


=== Benchmarking model: Baseline ===

--- Task: m4_hourly ---
Batch size: 20 | Inference time per sample: 0.0519 sec
Batch size: 40 | Inference time per sample: 0.0166 sec
Batch size: 60 | Inference time per sample: 0.0110 sec
Batch size: 80 | Inference time per sample: 0.0084 sec
Batch size: 100 | Inference time per sample: 0.0077 sec

Saved summaries → artifacts/Baseline_summaries.csv
Saved inference times → artifacts/Baseline_inference_times.csv


(  model_name                dataset_path dataset_config  horizon  num_windows  \
 0   Baseline  autogluon/chronos_datasets      m4_hourly       24            1   
 1   Baseline  autogluon/chronos_datasets      m4_hourly       24            1   
 2   Baseline  autogluon/chronos_datasets      m4_hourly       24            1   
 3   Baseline  autogluon/chronos_datasets      m4_hourly       24            1   
 4   Baseline  autogluon/chronos_datasets      m4_hourly       24            1   
 
    initial_cutoff  window_step_size  min_context_length max_context_length  \
 0             -24                24                   1               None   
 1             -24                24                   1               None   
 2             -24                24                   1               None   
 3             -24                24                   1               None   
 4             -24                24                   1               None   
 
    seasonality  ... static_co

ERROR:tornado.general:Uncaught exception in ZMQStream callback
Traceback (most recent call last):
  File "/home/lucas/mase/.venv/lib/python3.11/site-packages/zmq/eventloop/zmqstream.py", line 565, in _log_error
    f.result()
  File "/home/lucas/mase/.venv/lib/python3.11/site-packages/ipykernel/kernelbase.py", line 341, in dispatch_control
    await self.process_control(msg)
  File "/home/lucas/mase/.venv/lib/python3.11/site-packages/ipykernel/kernelbase.py", line 347, in process_control
    idents, msg = self.session.feed_identities(msg, copy=False)
                  ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/home/lucas/mase/.venv/lib/python3.11/site-packages/jupyter_client/session.py", line 993, in feed_identities
    raise ValueError(msg)
ValueError: DELIM not in msg_list
ERROR:tornado.general:Uncaught exception in ZMQStream callback
Traceback (most recent call last):
  File "/home/lucas/mase/.venv/lib/python3.11/site-packages/zmq/eventloop/zmqstream.py", line 565, in _l

In [11]:
import torch
import inspect
from types import SimpleNamespace

In [16]:
# Benchmark with Fused Time-Group Attention Triton Kernel
from chop.ir.graph.mase_graph import MaseGraph
from chop.passes.graph.transforms.fused_time_group_attention import fused_time_group_attention_pass
import torch

# Create MaseGraph and apply the transform pass
# We need some dummy inputs for tracing
dummy_context = torch.randn(20, 1440, device=DEVICE, dtype=torch.bfloat16)
# We trace the internal encoder model to apply the pass
mg = MaseGraph(model, hf_input_names=["context", "context_mask", "group_ids", "num_output_patches"])
mg, _ = fused_time_group_attention_pass(mg)

class FusedModelAdapter(torch.nn.Module):
    def __init__(self, compiled_model):
        super().__init__()
        self.model = compiled_model
        self.device = next(compiled_model.parameters()).device
        self.valid_kwargs = set(inspect.signature(compiled_model.forward).parameters.keys())
        self._printed_keys = False # Just to avoid spamming your console
        
    def forward(self, context, group_ids, **kwargs):
        incoming_args = {"context": context, "group_ids": group_ids, **kwargs}
        
        # 2. Ensure num_output_patches is passed through if requested by the pipeline
        if "context_mask" in self.valid_kwargs and "context_mask" not in incoming_args:
            incoming_args["context_mask"] = torch.zeros_like(context, dtype=torch.bool)
            
        filtered_args = {k: v for k, v in incoming_args.items() if k in self.valid_kwargs}
        
        output = self.model(**filtered_args)
        
        if isinstance(output, dict):
            # Debugging check to see what the compiled model is actually returning
            if not self._printed_keys:
                print(f"[DEBUG] Compiled model output keys: {list(output.keys())}")
                self._printed_keys = True
                
            if 'quantile_preds' not in output:
                # If 'quantile_preds' is missing, let's try to find a likely candidate 
                # instead of just blindly grabbing the first one.
                possible_keys = [k for k in output.keys() if 'pred' in k.lower() or 'out' in k.lower()]
                best_key = possible_keys[0] if possible_keys else list(output.keys())[0]
                output['quantile_preds'] = output[best_key]
                
            return SimpleNamespace(**output)
            
        return output

# Update the model in the pipeline with the fused graph
pipeline.model = FusedModelAdapter(mg.model)
pipeline.model.chronos_config = model.chronos_config

print("Running Fused Benchmark")
fev_bench_inference_time(
    model=pipeline.model,
    model_name="Fused_Triton",
    task_configs=tasks_configs,
    batch_sizes=BATCH_SIZES,
)


Done: 100%|██████████| 2093/2093 [00:00<00:00, 37358.14it/s]                                         


Running Fused Benchmark

=== Benchmarking model: Fused_Triton ===

--- Task: m4_hourly ---
[DEBUG] Compiled model output keys: ['quantile_preds', 'enc_time_self_attn_weights', 'enc_group_self_attn_weights']
Batch size: 20 | Inference time per sample: 0.0520 sec
Batch size: 40 | Inference time per sample: 0.0158 sec
Batch size: 60 | Inference time per sample: 0.0114 sec
Batch size: 80 | Inference time per sample: 0.0088 sec
Batch size: 100 | Inference time per sample: 0.0079 sec

Saved summaries → artifacts/Fused_Triton_summaries.csv
Saved inference times → artifacts/Fused_Triton_inference_times.csv


(     model_name                dataset_path dataset_config  horizon  \
 0  Fused_Triton  autogluon/chronos_datasets      m4_hourly       24   
 1  Fused_Triton  autogluon/chronos_datasets      m4_hourly       24   
 2  Fused_Triton  autogluon/chronos_datasets      m4_hourly       24   
 3  Fused_Triton  autogluon/chronos_datasets      m4_hourly       24   
 4  Fused_Triton  autogluon/chronos_datasets      m4_hourly       24   
 
    num_windows  initial_cutoff  window_step_size  min_context_length  \
 0            1             -24                24                   1   
 1            1             -24                24                   1   
 2            1             -24                24                   1   
 3            1             -24                24                   1   
 4            1             -24                24                   1   
 
   max_context_length  seasonality  ... static_columns  task_name test_error  \
 0               None            1  ...       